### Simple Gen AI APP using Langchain

In [ ]:
# https://docs.langchain.com/langsmith/deploy-self-hosted-full-platform
# https://docs.langchain.com/oss/python/langchain/studio#setup-local-agent-server

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
#Langsmith traching
os.environ["LangChain_API_KEY"] = os.getenv("LangChain_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "false"
os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")

In [3]:
# Data Ingestion - From the website we need to scrape the data
from langchain_community.document_loaders import WebBaseLoader

In [4]:
loader = WebBaseLoader("https://docs.langchain.com/langsmith/deploy-self-hosted-full-platform")
loader

In [7]:
docs = loader.load()
docs

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/deploy-self-hosted-full-platform', 'title': 'Enable LangSmith Deployment - Docs by LangChain', 'language': 'en'}, page_content='Enable LangSmith Deployment - Docs by LangChainSkip to main contentDocs by LangChain home pageLangSmithSearch...⌘KSupportGitHubTry LangSmithTry LangSmithSearch...NavigationSetup guidesEnable LangSmith DeploymentGet startedObservabilityEvaluationPrompt engineeringDeploymentPlatform setupReferenceOverviewSet up LangSmithCloud (SaaS)Self-hosted cloud architectureAWSAzureGCPHybridOverviewSetup guideSelf-hostedOverviewSetup guidesLangSmithEnable deploymentManage an installationConfigurationConnect external servicesPlatform auth & access controlSelf-hosted observabilityScripts for management tasksOn this pageOverviewPrerequisitesSetup(Optional) Configure additional data planesPrerequisitesDeploying to a different clusterDeploying to a different namespace in the same cluster(Optional) Configure authe

In [12]:
## After loading the data we need to divide the text into smaller chunks
## Then these chunks into vector embeddings and then store them in a vector database 
## example of vector embeddings using OpenAI and FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
texts = text_splitter.split_documents(docs)

In [13]:
## print the splitted texts
texts[:2]  # print first two chunks

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/deploy-self-hosted-full-platform', 'title': 'Enable LangSmith Deployment - Docs by LangChain', 'language': 'en'}, page_content='Enable LangSmith Deployment - Docs by LangChainSkip to main contentDocs by LangChain home pageLangSmithSearch...⌘KSupportGitHubTry LangSmithTry LangSmithSearch...NavigationSetup guidesEnable LangSmith DeploymentGet startedObservabilityEvaluationPrompt engineeringDeploymentPlatform setupReferenceOverviewSet up LangSmithCloud (SaaS)Self-hosted cloud architectureAWSAzureGCPHybridOverviewSetup guideSelf-hostedOverviewSetup guidesLangSmithEnable deploymentManage an installationConfigurationConnect external servicesPlatform auth & access controlSelf-hosted observabilityScripts for management tasksOn this pageOverviewPrerequisitesSetup(Optional) Configure additional data planesPrerequisitesDeploying to a different clusterDeploying to a different namespace in the same cluster(Optional) Configure authe

In [14]:
#embedding technique
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings()

In [15]:
from langchain_community.vectorstores import FAISS
#creating the vectorstore for the embeddings and texts
vectorstore = FAISS.from_documents(texts, embeddings)

In [16]:
#save the vectorstore locally


In [19]:
#query from a vectorstore db
query = "How to deploy Langchain self hosted full platform?"
docs = vectorstore.similarity_search(query)
docs[0].page_content  # print the most similar document



'for private registriesNext stepsSelf-hostedSetup guidesEnable LangSmith DeploymentCopy pageCopy pageThis guide shows you how to enable LangSmith Deployment on your self-hosted LangSmith instance. This adds a control plane and data plane that let you deploy, scale, and manage agents and applications directly through the LangSmith UI.'

In [30]:
# creating the llm model
from langchain_openai import OpenAI, ChatOpenAI
llm = ChatOpenAI(temperature=0.2, model_name="gpt-4o")

In [34]:
## Retrival Chain with LLM
#from langchain_core.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    
    """
    
    
    Answer the following question based only on the provided context:
    <context>
    {context}
    </context>

    """
    
)
chain = prompt|llm

In [38]:
retriver = vectorstore.as_retriever()